# 裂谷第一轮核验

原件只读。所有计算对应本目录冻结资料。这里重算关键记录数、资料异常和运动约束；完整扫描脚本位于 scripts。测试输入不具有天然参数身份。


In [1]:
from pathlib import Path
import csv, json, zipfile, hashlib, sys
sys.dont_write_bytecode = True
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
if not (root/'sources').exists(): root = root/'data_generation/stage/rifting'
assert (root/'sources').exists()
def csv_rows(path):
    with path.open(encoding='utf-8-sig', newline='') as f: return list(csv.DictReader(f))


In [2]:
wsm = csv_rows(root/'sources/new/N01/WSM_Database_2025.csv')
abc = [r for r in wsm if r['QUALITY'] in ['A','B','C']]
nf = [r for r in abc if r['REGIME']=='NF']
print({'WSM':len(wsm),'A-C':len(abc),'A-C NF':len(nf),'NF FMS':sum(r['TYPE']=='FMS' for r in nf)})
assert len(wsm)==100842 and len(nf)==22154


{'WSM': 100842, 'A-C': 77365, 'A-C NF': 22154, 'NF FMS': 21630}


In [3]:
with zipfile.ZipFile(root/'sources/new/N03/fault_statistics.zip') as z:
    prefix='Extracted fault statistics/Geometric_relationships/Appended_relationships_Model'
    b,e=z.read(prefix+'B.xlsx'),z.read(prefix+'E.xlsx')
print({'B_E_equal_bytes':b==e,'sha256':hashlib.sha256(b).hexdigest()})
assert b==e
faults=json.loads((root/'sources/new/N02/MSSM_faults.geojson').read_text(encoding='utf-8'))['features']
bad=[f['properties']['MSSM_id'] for f in faults if f['properties']['dip_lower']>f['properties']['dip_int']]
print({'MSSM_faults':len(faults),'dip_order_failure_ids':bad})
assert bad==['316']


{'B_E_equal_bytes': True, 'sha256': 'a45e5c2a00de0cf154a34620e88f8c88299d37f6dc143c2142f53b2604328ad5'}
{'MSSM_faults': 108, 'dip_order_failure_ids': ['316']}


In [4]:
import numpy as np
A=np.array([[-1.,1,0],[0,1,-1]])
d=np.array([-1.,-1.])
M=np.vstack([A,[.5,0,.5]])
u=np.linalg.solve(M,np.r_[d,0.])
print({'relative_constraint_rank':int(np.linalg.matrix_rank(A)),'unknowns':3,'velocities_mm_per_yr':u.tolist()})
assert np.allclose(u,[0,-1,0])


{'relative_constraint_rank': 2, 'unknowns': 3, 'velocities_mm_per_yr': [-0.0, -1.0, 0.0]}


## 计算范围

上面的块体例子检查相对约束、参考基准及共同中央状态。二维有限元完整检查见 scripts/method_checks.py 和 output/checks/method_checks.json。长期响应、三维断层生成与完整LEM尚未验证。
